# 최종 제출 재현 — 난임 임신성공 예측 · **RealMLP / TabM (5-fold)**

이 노트북을 **위에서 아래로 한 번 실행**하면 `submission.csv`가 생성됩니다.

**구성 — 4개 검증멤버 + 딥멤버 1개(게이트)**
`lgb(v2v3) · cat(v2v3) · xgb(v3) · lin(ratio) · **deep(RealMLP 또는 TabM)**`

> 🔬 **딥멤버 = from-scratch 신경망** (RealMLP-TD / TabM, `pytabkit`, Apache-2.0).
> TabICL·TabDPT 같은 **사전학습(in-context) 모델이 아님** — 네 데이터로 처음부터 학습.
> "TD = Tuned Defaults"는 *가중치*가 아니라 **하이퍼파라미터·전처리 기본값**이 튜닝됐다는 뜻.
> 학습 모델이므로 **다른 멤버와 동일하게 5-fold full OOF**로 생성(fold0 단축 아님).
> 네가 닫은 기본 임베딩-MLP NN과는 다른 계열(RealMLP=레시피, TabM=값싼 멀티헤드 앙상블) — "안 해본" 가벼운 멤버.

**셀 7 상단 `DEEP_MODEL`** 을 `"realmlp"` ↔ `"tabm"` 으로 바꿔 모델별 1회씩 실행.

**규정 준수 (test 누수 차단)**
- test는 **어떤 학습/전처리 fit에도 미포함.** RealMLP·TabM 내부 전처리(quantile·robust scaling)는
  각 fold의 `.fit()` 데이터(=train fold)에서만 적합 → **fold-내부·누수0**.
- OCC/AGE 순서맵·상수컬럼 제거·중앙값 대치는 **각 fold-train에서만** 산출. 외부데이터·유사라벨링 미사용.

**입력:** `train.csv` · `test.csv` · `sample_submission.csv`
**환경:** ★ GPU 세션 권장(딥멤버 학습, 5-fold ≈ 10~25분). 트리·선형은 CPU.
**산출:** `submission.csv` (+ 멤버 OOF/test = 누수안전 증빙)

In [2]:
# ============================================================
# 재현성 고정 (시드 · 결정성 · 버전 스냅샷)
# ============================================================
import os, random
import numpy as np
SEED = 42
os.environ["PYTHONHASHSEED"]="42"; os.environ["CUBLAS_WORKSPACE_CONFIG"]=":4096:8"; os.environ["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"
random.seed(SEED); np.random.seed(SEED)
NUM_THREADS = 8   # M1 성능코어 8개 활용(4→8, ~2배 가속). 트리 결정성: 이 스레드수(8) 고정 시 재실행 동일.
try:
    import torch
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
    torch.use_deterministic_algorithms(True)   # 엄격 결정성(임베딩 backward까지 결정적 경로 강제). 동일 환경서 NN bit-재현.
except Exception: pass
def snapshot_env(path="env_versions.json"):
    import json,sys,sklearn,scipy,pandas as pd
    v={"python":sys.version.split()[0],"numpy":np.__version__,"pandas":pd.__version__,
       "scikit-learn":sklearn.__version__,"scipy":scipy.__version__}
    for n in ["lightgbm","xgboost","catboost","torch"]:
        try: v[n]=__import__(n).__version__
        except Exception: v[n]=None
    try: json.dump(v,open(path,"w"),indent=2)
    except Exception: pass
    print("[env]",v); return v
ENV=snapshot_env()

[env] {'python': '3.13.13', 'numpy': '2.5.0', 'pandas': '3.0.3', 'scikit-learn': '1.9.0', 'scipy': '1.18.0', 'lightgbm': '4.6.0', 'xgboost': '3.3.0', 'catboost': '1.2.10', 'torch': '2.12.1'}


## 1. 데이터 로드 + 누수안전 피처 빌더 (행단위·train-only)

In [3]:
import glob, re, json
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from scipy.stats import rankdata
import lightgbm as lgb, xgboost as xgb
from catboost import CatBoostClassifier

def find_csv(n):
    # 로컬(data/) 우선 탐색 → 없으면 Kaggle(/kaggle/input) 폴백 (양쪽 환경 모두 동작)
    for d in ["./data","../data","../../data","../../../data","data"]:
        p=os.path.join(d,n)
        if os.path.exists(p): return p
    cur=os.path.abspath(os.getcwd())
    for _ in range(6):
        p=os.path.join(cur,"data",n)
        if os.path.exists(p): return p
        cur=os.path.dirname(cur)
    h=[p for p in glob.glob("/kaggle/input/**/*.csv",recursive=True) if os.path.basename(p)==n]
    assert h, f"{n} 없음 — 로컬 data/ 또는 Kaggle Add Input 확인"; return sorted(h,key=len)[0]
train=pd.read_csv(find_csv("train.csv")); test=pd.read_csv(find_csv("test.csv"))
TARGET="임신 성공 여부"; ID_COL="ID"; y=train[TARGET].astype(int).values; N=len(train)
print("train",train.shape,"| test",test.shape,"| base_rate=%.4f"%y.mean())

def NUM(df,c): return pd.to_numeric(df[c],errors="coerce") if c in df else pd.Series(np.nan,index=df.index)
def DIV(num,den): den=den.astype(float); return num.astype(float)/den.where(den>0)
def runr(x): return rankdata(x)/len(x)
COL_PROC="특정 시술 유형"; COL_RSN="배아 생성 주요 이유"
NOMINAL_COLS=["시술 시기 코드","시술 유형","배란 유도 유형","난자 출처","정자 출처"]
OCC=["총 시술 횟수","클리닉 내 총 시술 횟수","IVF 시술 횟수","DI 시술 횟수","총 임신 횟수","IVF 임신 횟수","DI 임신 횟수","총 출산 횟수","IVF 출산 횟수","DI 출산 횟수"]
AGE_MAPS={"시술 당시 나이":{"만18-34세":0,"만35-37세":1,"만38-39세":2,"만40-42세":3,"만43-44세":4,"만45-50세":5,"알 수 없음":-1},
 "난자 기증자 나이":{"만20세 이하":0,"만21-25세":1,"만26-30세":2,"만31-35세":3,"만36-40세":4,"만41-45세":5,"알 수 없음":-1},
 "정자 기증자 나이":{"만20세 이하":0,"만21-25세":1,"만26-30세":2,"만31-35세":3,"만36-40세":4,"만41-45세":5,"알 수 없음":-1}}
CMAP={"0회":0,"1회":1,"2회":2,"3회":3,"4회":4,"5회":5,"6회 이상":6}
_tp=lambda s: [] if pd.isna(s) else [t.strip() for t in re.split(r"[/:]",str(s)) if t.strip()]

# ── 트리 베이스 (명목형 카테고리 코드는 train 카테고리로만 fit) ──
def fit_tree(tr):
    st={}; ig={TARGET,ID_COL}
    st["dead"]=[c for c in tr.columns if c not in ig and tr[c].nunique(dropna=True)<=1]
    st["sparse"]=[c for c in tr.columns if c not in ig and c not in st["dead"] and tr[c].isna().mean()>0.98]
    st["lc"]={c:pd.Index(tr[c].astype("category").cat.categories) for c in NOMINAL_COLS if c in tr}
    st["pv"]=sorted({t for L in tr[COL_PROC].apply(_tp) for t in L}); return st
def tf_tree(df,st):
    df=df.copy()
    if TARGET in df: df=df.drop(columns=[TARGET])
    df["is_DI"]=(df["시술 유형"]=="DI").astype(int)
    df=df.drop(columns=[c for c in st["dead"] if c in df.columns])
    for c in st["sparse"]:
        if c in df: df[f"{c}_있음"]=df[c].notna().astype(int); df=df.drop(columns=[c])
    for c in OCC:
        if c in df: df[c]=df[c].astype(object).map(CMAP)
    for c,m in AGE_MAPS.items():
        if c in df: df[c]=df[c].astype(object).map(m)
    cats=[]
    for c,cc in st["lc"].items():
        if c in df: df[c]=pd.Categorical(df[c],categories=cc).codes.astype("int32"); cats.append(c)
    ts=df[COL_PROC].apply(_tp)
    for v in st["pv"]: df[f"proc_{v}"]=ts.apply(lambda L,v=v:int(v in L))
    df=df.drop(columns=[c for c in [COL_PROC,COL_RSN,ID_COL] if c in df.columns])
    obj=[c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]  # pandas3.0: str dtype 대비(is_numeric_dtype)
    if obj: df=df.drop(columns=obj)
    for c in cats: df[c]=df[c].fillna(-1).astype("int32")
    return df,[c for c in cats if c in df.columns]
st=fit_tree(train); Xb,CATF=tf_tree(train,st); Xb_te,_=tf_tree(test,st); Xb_te=Xb_te.reindex(columns=Xb.columns)
base_num=Xb.drop(columns=CATF); base_num_te=Xb_te.drop(columns=CATF)

# ── v2 게이팅 파생 (행단위) ──
def masks(df):
    return {"신선":NUM(df,"신선 배아 사용 여부")==1,"동결":NUM(df,"동결 배아 사용 여부")==1,
            "ICSI":NUM(df,"미세주입된 난자 수")>0,
            "본인난자":df["난자 출처"].astype(str)=="본인 제공","기증난자":df["난자 출처"].astype(str)=="기증 제공"}
def build_v2_gated(df):
    Mk=masks(df); F={}
    P1=DIV(NUM(df,"총 생성 배아 수"),NUM(df,"혼합된 난자 수")); P2=DIV(NUM(df,"미세주입에서 생성된 배아 수"),NUM(df,"미세주입된 난자 수"))
    P6=DIV(NUM(df,"총 생성 배아 수"),NUM(df,"수집된 신선 난자 수")); L3=NUM(df,"배아 이식 경과일")-NUM(df,"난자 혼합 경과일")
    F["g신선_수정률"]=P1.where(Mk["신선"]); F["gICSI_수정효율"]=P2.where(Mk["ICSI"])
    F["g본인_난자수율"]=P6.where(Mk["본인난자"]); F["g기증_난자수율"]=P6.where(Mk["기증난자"]); F["g신선_배양일수"]=L3.where(Mk["신선"])
    F["FZ1_동결해동이식간격"]=(NUM(df,"배아 이식 경과일")-NUM(df,"배아 해동 경과일")).where(Mk["동결"])
    F["FZ2_해동이식률"]=DIV(NUM(df,"이식된 배아 수"),NUM(df,"해동된 배아 수"))
    F["FZ3_해동난자수율"]=DIV(NUM(df,"총 생성 배아 수"),NUM(df,"해동 난자 수"))
    F["PG1_PGT강도"]=NUM(df,"착상 전 유전 검사 사용 여부").fillna(0)+NUM(df,"착상 전 유전 진단 사용 여부").fillna(0)
    F["PG2_PGT분류"]=NUM(df,"PGD 시술 여부").fillna(0)+NUM(df,"PGS 시술 여부").fillna(0)
    F["ST1_자극"]=NUM(df,"배란 자극 여부").fillna(0)
    return pd.DataFrame(F,index=df.index)
V2tr=build_v2_gated(train); V2te=build_v2_gated(test)

# ── 선형용 비율 파생 (행단위) ──
def build_lin_ratios(df):
    Mk=masks(df); F={}
    P1=DIV(NUM(df,"총 생성 배아 수"),NUM(df,"혼합된 난자 수")); P2=DIV(NUM(df,"미세주입에서 생성된 배아 수"),NUM(df,"미세주입된 난자 수"))
    P6=DIV(NUM(df,"총 생성 배아 수"),NUM(df,"수집된 신선 난자 수")); L3=NUM(df,"배아 이식 경과일")-NUM(df,"난자 혼합 경과일")
    F["P1_수정률"]=P1; F["P2_ICSI수정"]=P2; F["P6_난자수율"]=P6; F["L3_배양일수"]=L3
    F["P3_활용률"]=DIV(NUM(df,"이식된 배아 수")+NUM(df,"저장된 배아 수"),NUM(df,"총 생성 배아 수")); F["P5_저장률"]=DIV(NUM(df,"저장된 배아 수"),NUM(df,"총 생성 배아 수"))
    F["L6_배아perICSI난자"]=DIV(NUM(df,"총 생성 배아 수"),NUM(df,"미세주입된 난자 수")); F["N6_난자감모"]=DIV(NUM(df,"수집된 신선 난자 수")-NUM(df,"혼합된 난자 수"),NUM(df,"수집된 신선 난자 수"))
    F["g신선_수정률"]=P1.where(Mk["신선"]); F["gICSI_수정효율"]=P2.where(Mk["ICSI"]); F["g본인_수율"]=P6.where(Mk["본인난자"]); F["g기증_수율"]=P6.where(Mk["기증난자"]); F["g신선_배양일수"]=L3.where(Mk["신선"])
    F["FZ1_동결해동이식간격"]=(NUM(df,"배아 이식 경과일")-NUM(df,"배아 해동 경과일")).where(Mk["동결"]); F["FZ2_해동이식률"]=DIV(NUM(df,"이식된 배아 수"),NUM(df,"해동된 배아 수"))
    return pd.DataFrame(F,index=df.index)
RATtr=build_lin_ratios(train); RATte=build_lin_ratios(test)

# ── v3 신규 파생 (행단위) ──
def build_new_derived(df):
    F={}
    tx=NUM(df,"이식된 배아 수").fillna(0); sto=NUM(df,"저장된 배아 수").fillna(0); emb=NUM(df,"총 생성 배아 수").fillna(0)
    ses=(df["단일 배아 이식 여부"]==1).values
    F["EL_set_type"]=np.where(~ses,0,np.where(sto.values>0,2,1)).astype("int8")
    F["FA_no_transfer"]=(tx==0).astype("int8").values
    is_current=df[COL_RSN].astype(str).str.contains("현재 시술용",na=False).values
    F["FA_nontransfer_reason"]=(~is_current).astype("int8")
    return pd.DataFrame(F,index=df.index)
Dtr=build_new_derived(train); Dte=build_new_derived(test)

# ── v3 트리 후보 컬럼 (최종 선택 셋) ──
#   C1=배아이식 유형(EL_set_type), C2=무이식 플래그(FA_*).  C3(ICSI 경로효율)은 혼합난자⊇미세주입난자로 분모 오정의 → 영구 폐기.
V3_TREE_COLS = ["EL_set_type","FA_no_transfer","FA_nontransfer_reason"]
assert all(c in Dtr.columns for c in V3_TREE_COLS)
# 행단위 독립성(누수 0) 검증
assert np.array_equal(build_new_derived(train.head(300)).fillna(-9).values, Dtr.head(300).fillna(-9).values), "행단위 독립성 위반!"
print("피처 빌더 준비 ✅ | tf_tree",Xb.shape,"| v2게이팅",V2tr.shape[1],"| 비율",RATtr.shape[1],"| v3트리",len(V3_TREE_COLS))

train (256351, 69) | test (90067, 68) | base_rate=0.2583
피처 빌더 준비 ✅ | tf_tree (256351, 72) | v2게이팅 11 | 비율 15 | v3트리 3


## 2. 트리·선형 멤버 학습기 (fold-내부 fit · 결정적)

In [4]:
LGP=dict(objective="binary",metric="auc",verbose=-1,learning_rate=0.05,num_leaves=63,
         feature_fraction=0.8,bagging_fraction=0.8,bagging_freq=1,min_child_samples=50,
         deterministic=True,force_row_wise=True,num_threads=NUM_THREADS)
XGP=dict(objective="binary:logistic",eval_metric="auc",tree_method="hist",learning_rate=0.05,
         max_depth=6,subsample=0.8,colsample_bytree=0.8,nthread=NUM_THREADS)
TREE_ITERS=1500
def fit_one(kind,Xt,yt,Xv,yv,catf,seed):
    if kind=="lgb": return lgb.train(dict(LGP,seed=seed),lgb.Dataset(Xt,yt,categorical_feature=catf),TREE_ITERS,
        valid_sets=[lgb.Dataset(Xv,yv,categorical_feature=catf)],callbacks=[lgb.early_stopping(80,verbose=False),lgb.log_evaluation(0)])
    if kind=="xgb": return xgb.train(dict(XGP,seed=seed),xgb.DMatrix(Xt.values,label=yt),TREE_ITERS,
        evals=[(xgb.DMatrix(Xv.values,label=yv),"v")],early_stopping_rounds=80,verbose_eval=False)
    m=CatBoostClassifier(iterations=TREE_ITERS,learning_rate=0.05,depth=6,verbose=0,random_seed=seed,
        early_stopping_rounds=80,thread_count=NUM_THREADS); m.fit(Xt,yt,eval_set=(Xv,yv),cat_features=catf); return m
def pred_one(kind,m,X):
    if kind=="lgb": return m.predict(X)
    if kind=="xgb": return m.predict(xgb.DMatrix(X.values),iteration_range=(0,m.best_iteration+1))
    return m.predict_proba(X)[:,1]
def tree_member(kind, extra_tr, extra_te, seed=42):
    """단일 트리 모델 OOF + test (extra_* = 추가 파생 행렬). fold-내부 fit."""
    X  =pd.concat([Xb.reset_index(drop=True),    extra_tr.reset_index(drop=True)],axis=1)
    Xte=pd.concat([Xb_te.reset_index(drop=True), extra_te.reset_index(drop=True)],axis=1)
    folds=list(StratifiedKFold(5,shuffle=True,random_state=seed).split(X,y)); catf=[c for c in CATF if c in X.columns]
    o=np.zeros(N); tt=np.zeros(len(Xte))
    for tri,vai in folds:
        m=fit_one(kind,X.iloc[tri],y[tri],X.iloc[vai],y[vai],catf,seed); o[vai]=pred_one(kind,m,X.iloc[vai])
        tt+=pred_one(kind,m,Xte)/len(folds)
    return o,tt

# 선형 (타깃인코딩·스케일 전부 fold-내부 fit)
TE_COLS=NOMINAL_COLS+[COL_PROC,COL_RSN]
def te_fit(cat,yy,sm=20):
    g=pd.DataFrame({"c":cat.values,"y":yy}).groupby("c")["y"].agg(["mean","count"]); pr=float(yy.mean())
    return ((g["mean"]*g["count"]+pr*sm)/(g["count"]+sm)),pr
def lin_member(use_ratios=True,seed=42):
    oof=np.zeros(N); tst=np.zeros(len(test))
    for tri,vai in StratifiedKFold(5,shuffle=True,random_state=seed).split(base_num,y):
        Xt=base_num.iloc[tri].copy(); Xv=base_num.iloc[vai].copy(); Xte=base_num_te.copy()
        for c in TE_COLS:
            enc,pr=te_fit(train[c].astype(str).iloc[tri],y[tri])
            Xt[f"te_{c}"]=train[c].astype(str).iloc[tri].map(enc).fillna(pr).values
            Xv[f"te_{c}"]=train[c].astype(str).iloc[vai].map(enc).fillna(pr).values
            Xte[f"te_{c}"]=test[c].astype(str).map(enc).fillna(pr).values
        if use_ratios:
            Xt=pd.concat([Xt.reset_index(drop=True),RATtr.iloc[tri].reset_index(drop=True)],axis=1)
            Xv=pd.concat([Xv.reset_index(drop=True),RATtr.iloc[vai].reset_index(drop=True)],axis=1)
            Xte=pd.concat([Xte.reset_index(drop=True),RATte.reset_index(drop=True)],axis=1)
        med=Xt.median(); Xt=Xt.fillna(med); Xv=Xv.fillna(med)
        sc=StandardScaler().fit(Xt); m=LogisticRegression(max_iter=2000,C=0.5).fit(sc.transform(Xt),y[tri])
        oof[vai]=m.predict_proba(sc.transform(Xv))[:,1]
        tst+=m.predict_proba(sc.transform(Xte.fillna(med)))[:,1]/5
    return oof,tst
print("트리·선형 학습기 준비 ✅")

트리·선형 학습기 준비 ✅


## 3. 딥멤버 (RealMLP / TabM) — 5-fold full OOF (다른 멤버와 동일 fold)

from-scratch 학습 모델이라 트리처럼 **5-fold**: 각 fold마다 fold-train으로 학습 → fold-valid 예측(OOF 100% 채움), test는 fold별 예측 평균.
전처리·학습이 매 fold의 train에서만 적합되므로 **fold-내부·누수0**. (사전학습 모델의 fold0 단축과 달리, 학습 모델은 full OOF가 맞음.)

In [5]:
# ============================================================
# RealMLP / TabM — 5-fold full OOF (from-scratch, GPU 학습)
#   · from-scratch = 사전학습 아님. 매 fold의 .fit() 데이터(=fold-train)에서만 전처리·학습 → 누수 0.
#   · test는 어떤 fit에도 미포함. OCC/AGE 순서맵은 고정 사전(데이터 비의존) → 누수無.
#   · seed42 StratifiedKFold(5) — 다른 멤버와 동일 fold 정렬.
# ============================================================
import subprocess, sys, gc, time
subprocess.run([sys.executable,"-m","pip","install","-q","pytabkit"], check=False)
from pytabkit import RealMLP_TD_Classifier, TabM_D_Classifier
import torch
try: torch.use_deterministic_algorithms(False)   # MLP 학습 중 비결정적 CUDA op 허용(크래시 방지). random_state로 재현 근사.
except Exception: pass

DEEP_MODEL = "tabm"        # ★ 여기서 학습하는 딥멤버 (realmlp로도 변경 가능)
# 디바이스 우선순위: cuda > mps(애플 GPU) > cpu. MPS에서 미지원 연산으로 깨지면 deep_member가 cpu로 자동 폴백.
DEV = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("[deep]", DEEP_MODEL, "| device", DEV, "(MPS 실패 시 cpu 자동 폴백)")

def _deep_clf(seed, device=None):
    common=dict(device=(device or DEV), random_state=seed, n_cv=1, n_refit=0,
                val_metric_name="cross_entropy", verbosity=0)
    if DEEP_MODEL=="realmlp": return RealMLP_TD_Classifier(use_ls=False, **common)
    if DEEP_MODEL=="tabm":    return TabM_D_Classifier(**common)
    raise ValueError(DEEP_MODEL)

def _deep_base(df):
    D=df.drop(columns=[c for c in [TARGET,ID_COL] if c in df.columns]).copy()
    for c in OCC:
        if c in D: D[c]=D[c].astype(object).map(CMAP)
    for c,m in AGE_MAPS.items():
        if c in D: D[c]=D[c].astype(object).map(m)
    return D

def _deep_prep(fit_df, dfs):
    """전처리 통계(상수컬럼·중앙값)는 fit_df에서만 산출 → fold-내부/누수0.
       명목형은 문자열로 두고 cat_col_names(컬럼 이름)로 전달(모델이 내부 인코딩, 미지 카테고리도 처리)."""
    F=_deep_base(fit_df)
    keep=[c for c in F.columns if F[c].nunique(dropna=True)>1]            # 상수컬럼 제거(fit 기준)
    objk=[c for c in keep if not pd.api.types.is_numeric_dtype(F[c])]      # pandas3.0 str 대비
    numk=[c for c in keep if c not in objk]
    med={}
    for c in numk:
        mm=pd.to_numeric(F[c],errors="coerce").median(); med[c]=float(mm) if np.isfinite(mm) else 0.0
    outs=[]
    for df in dfs:
        D=_deep_base(df).reindex(columns=keep)
        for c in numk: D[c]=pd.to_numeric(D[c],errors="coerce").fillna(med[c]).astype(np.float32)
        for c in objk: D[c]=D[c].astype(str).fillna("NA")
        outs.append(D.reset_index(drop=True))
    cat_names=list(objk)
    return outs, cat_names

def deep_member(seed=42):
    """트리와 동일한 5-fold full OOF + test 평균. 전처리·학습 전부 fold-내부.
       MPS에서 미지원 연산으로 깨지면 그 시점부터 cpu로 자동 폴백(이후 fold도 cpu)."""
    global DEV
    folds=list(StratifiedKFold(5,shuffle=True,random_state=seed).split(train,y))
    oof=np.full(N,np.nan); tt=np.zeros(len(test)); _t0=time.perf_counter()
    for fi,(tri,vai) in enumerate(folds):
        (Xt,Xv,Xte),cat_names=_deep_prep(train.iloc[tri],[train.iloc[tri],train.iloc[vai],test])  # 전처리 fit = fold-train만
        try:
            clf=_deep_clf(seed,DEV); clf.fit(Xt, y[tri], cat_col_names=cat_names)                    # ★ 학습 = fold-train ONLY
        except Exception as e:
            if DEV=="cpu": raise
            print(f"    [경고] device={DEV} 실패 → cpu 폴백 ({type(e).__name__}: {str(e)[:80]})", flush=True)
            DEV="cpu"; clf=_deep_clf(seed,DEV); clf.fit(Xt, y[tri], cat_col_names=cat_names)
        oof[vai]=clf.predict_proba(Xv)[:,1]
        tt += clf.predict_proba(Xte)[:,1]/len(folds)
        print(f"    fold{fi} AUC={roc_auc_score(y[vai],oof[vai]):.5f} | dev={DEV} | 누적 {(time.perf_counter()-_t0)/60:.1f}분", flush=True)
        del clf, Xt, Xv, Xte; gc.collect()
        try:
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        except Exception: pass
    return oof, tt

oof_deep, test_deep = deep_member(seed=42)
print(f"{DEEP_MODEL} 5-fold full OOF AUC = {roc_auc_score(y,oof_deep):.5f}")

[deep] tabm | device mps (MPS 실패 시 cpu 자동 폴백)
    fold0 AUC=0.73700 | dev=mps | 누적 9.8분
    fold1 AUC=0.74180 | dev=mps | 누적 18.2분
    fold2 AUC=0.74044 | dev=mps | 누적 27.0분
    fold3 AUC=0.73744 | dev=mps | 누적 34.2분
    fold4 AUC=0.73876 | dev=mps | 누적 45.4분
tabm 5-fold full OOF AUC = 0.73905


## 4. 5개 멤버 생성 (seed42 · 동일 fold)

In [6]:
members_oof={}; members_test={}
# ── lgb·cat·xgb·lin = 외부 환경에서 만든 OOF/test 파일 불러오기 (로컬 학습 생략) ──
#    포맷: oof_{m}.csv=[oof_{m}, y] · test_{m}.csv=[ID, test_{m}]  (이 노트북 저장 포맷과 동일)
#    find_csv가 data/ 폴더에서 찾음. y(라벨)·ID 정렬을 assert로 검증 → 행 어긋남 즉시 탐지.
def _load_oof(m):
    d=pd.read_csv(find_csv(f"oof_{m}.csv")); col=f"oof_{m}"
    assert col in d.columns, f"oof_{m}.csv 컬럼 이상: {list(d.columns)}"
    assert len(d)==N, f"oof_{m}.csv 행수 {len(d)}≠train {N}"
    if "y" in d.columns: assert np.array_equal(d["y"].astype(int).values,y), f"oof_{m}.csv y 불일치 — 행정렬 깨짐!"
    return d[col].values
def _load_test(m):
    d=pd.read_csv(find_csv(f"test_{m}.csv")); col=f"test_{m}"
    assert col in d.columns, f"test_{m}.csv 컬럼 이상: {list(d.columns)}"
    if "ID" in d.columns:                       # ID 기준으로 test 순서에 맞춰 재정렬(정렬 어긋나도 안전)
        out=d.set_index("ID")[col].reindex(test[ID_COL].values).values
        assert not np.isnan(out).any(), f"test_{m}.csv: test ID와 매칭 안 되는 행 존재"
        return out
    assert len(d)==len(test), f"test_{m}.csv 행수 {len(d)}≠test {len(test)}"
    return d[col].values

for m in ["lgb","cat","xgb","lin"]:
    members_oof[m]=_load_oof(m); members_test[m]=_load_test(m)
    print(f"  [파일] {m}  OOF={roc_auc_score(y,members_oof[m]):.5f}")

# ── deep(tabm) = 이 노트북에서 학습한 결과 (셀 7) ──
members_oof[DEEP_MODEL]=oof_deep; members_test[DEEP_MODEL]=test_deep
print(f"  [로컬] {DEEP_MODEL} full OOF={roc_auc_score(y,oof_deep):.5f}")

# 로컬 학습한 deep 멤버만 저장(불러온 4개 파일은 원본 보존)
pd.DataFrame({f"oof_{DEEP_MODEL}":oof_deep,"y":y}).to_csv(f"oof_{DEEP_MODEL}.csv",index=False)
pd.DataFrame({"ID":test[ID_COL].values,f"test_{DEEP_MODEL}":test_deep}).to_csv(f"test_{DEEP_MODEL}.csv",index=False)
print(f"멤버 준비 완료 ✅ (lgb·cat·xgb·lin=파일 / {DEEP_MODEL}=로컬학습)")

  [파일] lgb  OOF=0.73965
  [파일] cat  OOF=0.73974
  [파일] xgb  OOF=0.73972
  [파일] lin  OOF=0.72029
  [로컬] tabm full OOF=0.73905
멤버 준비 완료 ✅ (lgb·cat·xgb·lin=파일 / tabm=로컬학습)


## 5. 최종 블렌드 → `submission.csv`

모든 멤버가 **full OOF**이므로 게이트도 전체 OOF에서 증분을 잰다(fold0 슬라이스 아님 → 통계력 ↑).
딥멤버 증분 paired-bootstrap CI 하한>0 이면 5멤버, 아니면 검증된 4멤버로 자동 폴백.

In [7]:
from scipy.stats import spearmanr
def hill(d,yy,n=120):
    nm=list(d); s0={k:roc_auc_score(yy,d[k]) for k in nm}; b=max(s0,key=s0.get)
    ens=[b]; s=d[b].copy(); best=(list(ens),s0[b])
    for _ in range(n):
        cb,ca=None,-1
        for k in nm:
            a=roc_auc_score(yy,(s+d[k])/(len(ens)+1))
            if a>ca: ca,cb=a,k
        ens.append(cb); s=s+d[cb]
        if ca>best[1]: best=(list(ens),ca)
    from collections import Counter; c=Counter(best[0]); return {k:c.get(k,0)/len(best[0]) for k in nm},best[1]

BASE=["lgb","cat","xgb","lin"]; ALL=BASE+[DEEP_MODEL]
for m in ALL: assert roc_auc_score(y,members_oof[m])<0.999, f"{m} 누수 의심"

# 4멤버 기준 vs 5멤버 (전체 OOF — 모두 full이라 같은 지반)
R ={m:runr(members_oof[m]) for m in ALL}
w4,oof4=hill({m:R[m] for m in BASE}, y)
w5,oof5=hill(R, y)
print(f"[full OOF] 4멤버 {oof4:.5f} → +{DEEP_MODEL} {oof5:.5f}  (Δ={oof5-oof4:+.5f})")
print("  ρ:", {k:round(spearmanr(members_oof[DEEP_MODEL],members_oof[k]).correlation,3) for k in BASE})

# 딥멤버 증분 paired-bootstrap CI (전체 OOF에서)
rng=np.random.RandomState(SEED); n=N
pb=sum(w4[k]*R[k] for k in w4 if w4[k]>0); pp=sum(w5[k]*R[k] for k in w5 if w5[k]>0)
d=np.empty(2000)
for i in range(2000):
    bi=rng.randint(0,n,n); d[i]=roc_auc_score(y[bi],pp[bi])-roc_auc_score(y[bi],pb[bi])
lo,hi=np.percentile(d,[2.5,97.5])
INCLUDE_DEEP = lo>0
print(f"  증분 95% CI=[{lo:+.5f},{hi:+.5f}] → {DEEP_MODEL+' 포함 ✅' if INCLUDE_DEEP else DEEP_MODEL+' 제외(4멤버)'}")

w = w5 if INCLUDE_DEEP else w4
p_test=sum(w[m]*runr(members_test[m]) for m in w if w[m]>0)
assert not np.isnan(p_test).any()
try: sp=[find_csv("sample_submission.csv")]   # 로컬 data/ 우선, Kaggle 폴백
except Exception: sp=[]
if sp:
    s=pd.read_csv(sp[0]); pc=[c for c in s.columns if c.lower()!="id"][0]; s[pc]=p_test
else:
    s=pd.DataFrame({"ID":test[ID_COL].values,"probability":p_test}); pc="probability"
s.to_csv("submission.csv",index=False)
print(f"💾 submission.csv | n={len(s)} | {DEEP_MODEL} {'포함' if INCLUDE_DEEP else '제외'}")
print("   가중치 →", {k:round(v,3) for k,v in w.items()})

[full OOF] 4멤버 0.74044 → +tabm 0.74064  (Δ=+0.00021)
  ρ: {'lgb': np.float64(0.98), 'cat': np.float64(0.98), 'xgb': np.float64(0.982), 'lin': np.float64(0.901)}
  증분 95% CI=[+0.00008,+0.00031] → tabm 포함 ✅
💾 submission.csv | n=90067 | tabm 포함
   가중치 → {'lgb': 0.25, 'cat': 0.262, 'xgb': 0.19, 'lin': 0.036, 'tabm': 0.262}


In [13]:
# ============================================================
# [추가 셀] nn·realmlp 합류 → 0.74219 베이스(5멤버) 대비 후보 게이트 (전부 CPU)
#   · 네 노트북의 find_csv() 재사용(= data/ 폴더 탐색). 컬럼순서 무관·ID 정렬.
#   · members_oof엔 이미 lgb·cat·xgb·lin·tabm 있음 → nn·realmlp만 추가.
# ============================================================
from scipy.stats import spearmanr

def _add_member(m):
    if m in members_oof: return True
    try:
        op=find_csv(f"oof_{m}.csv"); tp=find_csv(f"test_{m}.csv")
    except Exception as e:
        print(f"  [건너뜀] {m} (find_csv 실패: {e})"); return False
    od=pd.read_csv(op); td=pd.read_csv(tp)
    o=od[f"oof_{m}"].values
    assert len(o)==N, f"oof_{m} 행수 {len(o)}≠{N}"
    if "y" in od.columns:
        assert np.array_equal(od["y"].astype(int).values, y), f"oof_{m} y 불일치 — 행정렬 깨짐!"
    t=td.set_index("ID")[f"test_{m}"].reindex(test[ID_COL].values).values   # ID 정렬·컬럼순서 무관
    assert not np.isnan(t).any(), f"test_{m}: test ID 매칭 실패"
    a=roc_auc_score(y,o); assert a<0.999, f"{m} 누수 의심 AUC={a}"
    members_oof[m]=o; members_test[m]=t
    print(f"  [로드] {m:8s} oof={a:.5f}  ←  {op}")
    return True

for nm in ["nn","realmlp"]:
    _add_member(nm)

# ---- 힐클라이밍 ----
def hill(d,yy,n=120):
    nm=list(d); s0={k:roc_auc_score(yy,d[k]) for k in nm}; b=max(s0,key=s0.get)
    ens=[b]; s=d[b].copy(); best=(list(ens),s0[b])
    for _ in range(n):
        cb,ca=None,-1
        for k in nm:
            a=roc_auc_score(yy,(s+d[k])/(len(ens)+1))
            if a>ca: ca,cb=a,k
        ens.append(cb); s=s+d[cb]
        if ca>best[1]: best=(list(ens),ca)
    from collections import Counter; c=Counter(best[0]); return {k:c.get(k,0)/len(best[0]) for k in nm},best[1]

R={m:runr(members_oof[m]) for m in members_oof}
def blend_oof(keys): w,a=hill({k:R[k] for k in keys},y); return w,a
def ci_vs_base(base_keys,cand_keys,iters=2000):
    wb,ab=blend_oof(base_keys); wc,ac=blend_oof(cand_keys)
    pb=sum(wb[k]*R[k] for k in wb if wb[k]>0); pc=sum(wc[k]*R[k] for k in wc if wc[k]>0)
    rng=np.random.RandomState(SEED); n=N; d=np.empty(iters)
    for i in range(iters):
        bi=rng.randint(0,n,n); d[i]=roc_auc_score(y[bi],pc[bi])-roc_auc_score(y[bi],pb[bi])
    lo,hi=np.percentile(d,[2.5,97.5]); return ab,ac,ac-ab,lo,hi,wc

# ---- 베이스(5멤버) 정합성 ----
BASE5=["lgb","cat","xgb","lin","nn"]
assert all(k in members_oof for k in BASE5), f"누락: {[k for k in BASE5 if k not in members_oof]}"
wb5,ob5=blend_oof(BASE5)
print(f"\n[베이스 5멤버 {{lgb,cat,xgb,lin,nn}}] OOF={ob5:.5f}  (0.74067 근처면 정합 OK)")
print("  가중치:", {k:round(v,3) for k,v in wb5.items()})

# ---- 후보 게이트 ----
have=lambda *ks: all(k in members_oof for k in ks)
cands=[]
if have("tabm"):           cands.append(("+tabm (6)",         BASE5+["tabm"]))
if have("realmlp"):        cands.append(("+realmlp (6)",      BASE5+["realmlp"]))
if have("tabm","realmlp"): cands.append(("+tabm+realmlp (7)", BASE5+["tabm","realmlp"]))
if have("tabm"):           cands.append(("nn→tabm 교체",       ["lgb","cat","xgb","lin","tabm"]))
if have("realmlp"):        cands.append(("nn→realmlp 교체",    ["lgb","cat","xgb","lin","realmlp"]))

print(f"\n{'조합':<20}{'OOF':>9}{'Δ vs base':>12}{'95% CI':>22}  판정")
print("-"*70)
results=[]
for label,keys in cands:
    ab,ac,delta,lo,hi,wc=ci_vs_base(BASE5,keys); ok=lo>0
    print(f"{label:<20}{ac:>9.5f}{delta:>+12.5f}   [{lo:+.5f},{hi:+.5f}]  {'✅' if ok else '❌'}")
    results.append((label,keys,ac,delta,lo,hi,wc,ok))

# ---- '추가형' 중 CI 통과 최선 1개로 제출 ----
passed=[r for r in results if r[7] and "교체" not in r[0]]
pick=max(passed,key=lambda r:r[2]) if passed else None
if pick is None:
    print("\n→ CI 0 넘는 '추가' 조합 없음. 0.74219 베이스(5멤버) 유지 권장. 제출파일 미생성.")
else:
    label,keys,ac,delta,lo,hi,wc,_=pick
    p_test=sum(wc[k]*runr(members_test[k]) for k in wc if wc[k]>0)
    assert not np.isnan(p_test).any()
    pd.DataFrame({"ID":test[ID_COL].values,"probability":p_test}).to_csv("submission_best.csv",index=False)
    print(f"\n💾 submission_best.csv ← {label} | OOF={ac:.5f} (Δ{delta:+.5f}, CI 하한{lo:+.5f})")
    print("   가중치:", {k:round(v,3) for k,v in wc.items()})

  [로드] nn       oof=0.73751  ←  ../../data/oof_nn.csv
  [로드] realmlp  oof=0.73918  ←  ../../data/oof_realmlp.csv

[베이스 5멤버 {lgb,cat,xgb,lin,nn}] OOF=0.74067  (0.74067 근처면 정합 OK)
  가중치: {'lgb': 0.278, 'cat': 0.259, 'xgb': 0.222, 'lin': 0.028, 'nn': 0.213}

조합                        OOF   Δ vs base                95% CI  판정
----------------------------------------------------------------------
+tabm (6)             0.74076    +0.00009   [+0.00001,+0.00016]  ✅
+realmlp (6)          0.74073    +0.00006   [-0.00001,+0.00012]  ❌
+tabm+realmlp (7)     0.74077    +0.00010   [+0.00002,+0.00017]  ✅
nn→tabm 교체            0.74064    -0.00003   [-0.00016,+0.00011]  ❌
nn→realmlp 교체         0.74057    -0.00010   [-0.00024,+0.00003]  ❌

💾 submission_best.csv ← +tabm+realmlp (7) | OOF=0.74077 (Δ+0.00010, CI 하한+0.00002)
   가중치: {'lgb': 0.224, 'cat': 0.207, 'xgb': 0.164, 'lin': 0.026, 'nn': 0.155, 'tabm': 0.147, 'realmlp': 0.078}


In [14]:
# ---- '추가형' 중 CI 통과 최선 1개로 제출 ----
passed=[r for r in results if r[7] and "교체" not in r[0] and "realmlp" not in r[0]]  # realmlp 제외 → +tabm(6) 선택
pick=max(passed,key=lambda r:r[2]) if passed else None
if pick is None:
    print("\n→ CI 0 넘는 '추가' 조합 없음. 0.74219 베이스(5멤버) 유지 권장. 제출파일 미생성.")
else:
    label,keys,ac,delta,lo,hi,wc,_=pick
    p_test=sum(wc[k]*runr(members_test[k]) for k in wc if wc[k]>0)
    assert not np.isnan(p_test).any()
    pd.DataFrame({"ID":test[ID_COL].values,"probability":p_test}).to_csv("submission_best.csv",index=False)
    print(f"\n💾 submission_best.csv ← {label} | OOF={ac:.5f} (Δ{delta:+.5f}, CI 하한{lo:+.5f})")
    print("   가중치:", {k:round(v,3) for k,v in wc.items()})


💾 submission_best.csv ← +tabm (6) | OOF=0.74076 (Δ+0.00009, CI 하한+0.00001)
   가중치: {'lgb': 0.23, 'cat': 0.23, 'xgb': 0.172, 'lin': 0.023, 'nn': 0.161, 'tabm': 0.184}
